# Day 15 — Silver Streaming Transformation: vehicle_battery_live
**Source:** `bronze/event-stream/vehicle_battery_live/` (Delta, append-only)  
**Sink:** `silver/sl_vehicle_battery_live/` (Delta, MERGE upsert)  
**Quarantine:** `silver/quarantine/vehicle_battery_invalid/` (Delta, append)  
**Checkpoint:** `silver/_checkpoints/vehicle-battery-live/`  

### What this notebook does (production pattern)
1. Reads Bronze Delta as a stream — picks up every new append automatically
2. Applies 9 DQ rules per record — any failure routes the row to quarantine
3. Deduplicates by `event_id` within each micro-batch (idempotent replay-safe)
4. MERGEs clean records into Silver — upsert on `event_id` (exactly-once)
5. APPENDs rejected records to quarantine table with pipe-separated failure reasons

**Run cells 1 to 7 in order. Keep the Day 14 Bronze stream running in its own notebook.**

In [ ]:
# Cell 1: Load secrets
SP_CLIENT_ID     = dbutils.secrets.get(scope='kv-ev-scope', key='sp-client-id')
SP_CLIENT_SECRET = dbutils.secrets.get(scope='kv-ev-scope', key='sp-client-secret')
SP_TENANT_ID     = dbutils.secrets.get(scope='kv-ev-scope', key='sp-tenant-id')
STORAGE_ACCOUNT  = dbutils.secrets.get(scope='kv-ev-scope', key='adls-account-name')
print('Secrets loaded.')

In [ ]:
# Cell 2: Configure ADLS Gen2 OAuth
spark.conf.set(f'fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net', 'OAuth')
spark.conf.set(f'fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net',
               'org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider')
spark.conf.set(f'fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net', SP_CLIENT_ID)
spark.conf.set(f'fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net', SP_CLIENT_SECRET)
spark.conf.set(f'fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net',
               f'https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token')
print(f'ADLS OAuth configured: {STORAGE_ACCOUNT}')

In [ ]:
# Cell 3: Define paths
BRONZE_PATH     = f'abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/event-stream/vehicle_battery_live/'
SILVER_PATH     = f'abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/sl_vehicle_battery_live/'
QUARANTINE_PATH = f'abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/quarantine/vehicle_battery_invalid/'
CHECKPOINT_PATH = f'abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/vehicle-battery-live/'

print(f'Bronze:     {BRONZE_PATH}')
print(f'Silver:     {SILVER_PATH}')
print(f'Quarantine: {QUARANTINE_PATH}')
print(f'Checkpoint: {CHECKPOINT_PATH}')

In [ ]:
# Cell 4: Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lit, current_timestamp, when, concat_ws
from delta.tables import DeltaTable
print('Imports done.')

In [ ]:
# Cell 5: Silver transformation — foreachBatch function
#
# DQ rules applied per row:
#   R01  event_id not null            (dedup key)
#   R02  vehicle_id not null
#   R03  session_id not null
#   R04  battery_pct: 0.0 to 100.0   (Li-ion physics)
#   R05  charging_rate_kw: 0 to 350  (max real-world charger = 350 kW)
#   R06  battery_temp_c: -10 to 80   (Li-ion safe operating range)
#   R07  state_of_charge_target: 1 to 100
#   R08  event_ts not null
#   R09  _is_corrupt not True         (Bronze JSON parse failed)

def transform_to_silver(batch_df, batch_id):
    row_count = batch_df.count()
    if row_count == 0:
        print(f'[Batch {batch_id}] Empty batch — skipping.')
        return

    # Step 1: Tag each row with DQ failure reasons
    dq_df = (
        batch_df
        .withColumn('_r01', when(col('event_id').isNull(), lit('NULL_event_id')).otherwise(lit(None)))
        .withColumn('_r02', when(col('vehicle_id').isNull(), lit('NULL_vehicle_id')).otherwise(lit(None)))
        .withColumn('_r03', when(col('session_id').isNull(), lit('NULL_session_id')).otherwise(lit(None)))
        .withColumn('_r04', when(
            col('battery_pct').isNull() | (col('battery_pct') < 0.0) | (col('battery_pct') > 100.0),
            lit('INVALID_battery_pct')).otherwise(lit(None)))
        .withColumn('_r05', when(
            col('charging_rate_kw').isNull() | (col('charging_rate_kw') <= 0.0) | (col('charging_rate_kw') > 350.0),
            lit('INVALID_charging_rate_kw')).otherwise(lit(None)))
        .withColumn('_r06', when(
            col('battery_temp_c').isNull() | (col('battery_temp_c') < -10.0) | (col('battery_temp_c') > 80.0),
            lit('INVALID_battery_temp_c')).otherwise(lit(None)))
        .withColumn('_r07', when(
            col('state_of_charge_target_pct').isNull() |
            (col('state_of_charge_target_pct') < 1) | (col('state_of_charge_target_pct') > 100),
            lit('INVALID_soc_target')).otherwise(lit(None)))
        .withColumn('_r08', when(col('event_ts').isNull(), lit('NULL_event_ts')).otherwise(lit(None)))
        .withColumn('_r09', when(col('_is_corrupt') == True, lit('CORRUPT_bronze_json')).otherwise(lit(None)))
        .withColumn('_dq_reasons', concat_ws('|',
            col('_r01'), col('_r02'), col('_r03'), col('_r04'),
            col('_r05'), col('_r06'), col('_r07'), col('_r08'), col('_r09')))
        .withColumn('_dq_passed',
            when(col('_dq_reasons') == '', lit('true')).otherwise(lit('false')))
    )

    # Step 2: Split clean vs bad
    clean_df = dq_df.filter(col('_dq_passed') == 'true')
    bad_df   = dq_df.filter(col('_dq_passed') == 'false')

    # Step 3: Dedup within batch — keep latest ingestion per event_id
    dedup_window = Window.partitionBy('event_id').orderBy(col('_ingestion_ts').desc())
    clean_deduped = (
        clean_df
        .withColumn('_rn', F.row_number().over(dedup_window))
        .filter(col('_rn') == 1)
        .drop('_rn', '_r01', '_r02', '_r03', '_r04', '_r05', '_r06', '_r07', '_r08', '_r09', '_dq_reasons')
        .withColumn('_silver_ingested_at', current_timestamp())
        .select('event_id', 'vehicle_id', 'session_id', 'station_id', 'charger_id',
                'battery_pct', 'charging_rate_kw', 'battery_temp_c',
                'state_of_charge_target_pct', 'estimated_minutes_to_full',
                'event_ts', 'event_date', '_silver_ingested_at', '_dq_passed')
    )

    # Step 4: MERGE into Silver — upsert on event_id (idempotent)
    clean_count = clean_deduped.count()
    if clean_count > 0:
        if DeltaTable.isDeltaTable(spark, SILVER_PATH):
            silver_table = DeltaTable.forPath(spark, SILVER_PATH)
            (
                silver_table.alias('silver')
                .merge(clean_deduped.alias('batch'), 'silver.event_id = batch.event_id')
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
        else:
            # First run — create Silver Delta table
            (
                clean_deduped.write
                .format('delta').mode('overwrite')
                .partitionBy('event_date')
                .save(SILVER_PATH)
            )

    # Step 5: Write quarantine
    bad_count = bad_df.count()
    if bad_count > 0:
        (
            bad_df
            .withColumn('_quarantine_ts', current_timestamp())
            .withColumn('_batch_id', lit(batch_id))
            .select('event_id', 'vehicle_id', 'battery_pct', 'charging_rate_kw',
                    'battery_temp_c', 'event_ts', '_dq_reasons', '_quarantine_ts', '_batch_id', '_source')
            .write.format('delta').mode('append').save(QUARANTINE_PATH)
        )

    print(f'[Batch {batch_id}] In: {row_count} | Clean merged: {clean_count} | Quarantined: {bad_count}')

print('transform_to_silver() defined.')

In [ ]:
# Cell 6: Start Silver streaming query
# Reads Bronze Delta as a stream. trigger=60s gives Bronze (30s) time to settle each cycle.
# foreachBatch enables MERGE semantics not possible with native writeStream.format('delta').

silver_query = (
    spark.readStream
    .format('delta')
    .option('ignoreChanges', 'true')
    .load(BRONZE_PATH)
    .writeStream
    .foreachBatch(transform_to_silver)
    .trigger(processingTime='60 seconds')
    .option('checkpointLocation', CHECKPOINT_PATH)
    .start()
)

print(f'Silver stream started. ID: {silver_query.id}')
print(f'Trigger: 60s | Bronze -> DQ (9 rules) + dedup + MERGE -> Silver')
print(f'Silver:     {SILVER_PATH}')
print(f'Quarantine: {QUARANTINE_PATH}')

In [ ]:
# Cell 7: Monitor Silver stream (cancel anytime — stream keeps running)
import time
print('Silver stream live. Cancel to stop monitoring.\n')
while silver_query.isActive:
    p = silver_query.lastProgress
    if p:
        print(f'[Batch {p.get("batchId","?")}] '
              f'Input: {p.get("numInputRows",0):,} rows | '
              f'Processing rate: {p.get("processedRowsPerSecond",0.0):.1f} rows/sec')
    else:
        print('Waiting for first Silver batch...')
    time.sleep(60)
print('Silver stream stopped.')

In [ ]:
# Cell 8 (OPTIONAL): Verify Silver and quarantine
# Run in a separate notebook or after cancelling Cell 7.

from pyspark.sql import functions as F

silver_df = spark.read.format('delta').load(SILVER_PATH)
print(f'Total Silver rows: {silver_df.count():,}')

print('\nBattery % range per vehicle:')
silver_df.groupBy('vehicle_id').agg(
    F.min('battery_pct').alias('min_pct'),
    F.max('battery_pct').alias('max_pct'),
    F.avg('battery_pct').alias('avg_pct'),
    F.count('event_id').alias('events')
).orderBy('vehicle_id').show()

print('\nQuarantine summary:')
try:
    q = spark.read.format('delta').load(QUARANTINE_PATH)
    print(f'Total quarantined: {q.count():,}')
    q.groupBy('_dq_reasons').count().orderBy('count', ascending=False).show()
except Exception as e:
    print(f'No quarantine records yet — all rows passed DQ.')